## Step 1: Environment Setup

**⏱️ Time: ~10 minutes**

This cell will:
- Check GPU availability
- Clone the repository
- Install all dependencies
- Mount Google Drive

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be VERY slow.")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify data exists
import os
data_path = '/content/drive/MyDrive/Topo-Brain-Data/Nifti'
if os.path.exists(data_path):
    subjects = [d for d in os.listdir(data_path) if d.startswith('sub-')]
    print(f"✓ Found {len(subjects)} subjects in Google Drive")
    print(f"  Subjects: {subjects[:5]}..." if len(subjects) > 5 else f"  Subjects: {subjects}")
else:
    print(f"❌ Data not found at: {data_path}")
    print("Please upload your Nifti folder to Google Drive: MyDrive/Topo-Brain-Data/Nifti/")
    raise FileNotFoundError(f"Data directory not found: {data_path}")

In [ ]:
# Clone repository (or use existing code)
import os

# Check if repo already exists
if not os.path.exists('/content/Topo-Brain'):
    print("Cloning repository...")
    !git clone https://github.com/prabeshx12/Topo-Brain.git /content/Topo-Brain
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

# Change to project directory
os.chdir('/content/Topo-Brain')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies... (this may take 5-10 minutes)")

# Install PyTorch with CUDA support (if not already installed)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install core dependencies
!pip install -q monai nibabel SimpleITK scipy scikit-image
!pip install -q matplotlib seaborn tqdm pyyaml
!pip install -q tensorboard

# Install topology library (CRITICAL)
!pip install -q gudhi

# Install perceptual loss dependencies
!pip install -q lpips pytorch-fid

# Install HD-BET for skull stripping
!cd /content/Topo-Brain/HD-BET && pip install -q -e .

print("\n✓ All dependencies installed!")

# Verify critical imports
try:
    import gudhi
    print("✓ GUDHI installed successfully")
except ImportError:
    print("❌ GUDHI installation failed - trying conda...")
    !conda install -c conda-forge gudhi -y

In [ ]:
# Create symbolic links to Google Drive data
import os
import shutil

# Link data directory
drive_data = '/content/drive/MyDrive/Topo-Brain-Data'
local_nifti = '/content/Topo-Brain/Nifti'

if os.path.exists(local_nifti):
    print("Removing existing Nifti link...")
    os.remove(local_nifti) if os.path.islink(local_nifti) else shutil.rmtree(local_nifti)

print("Creating symbolic link to Google Drive data...")
os.symlink(f'{drive_data}/Nifti', local_nifti)

# Create output directories in Google Drive (to persist between sessions)
output_dirs = [
    f'{drive_data}/preprocessed',
    f'{drive_data}/preprocessed_registered',
    f'{drive_data}/checkpoints',
    f'{drive_data}/visualizations',
    f'{drive_data}/logs'
]

for dir_path in output_dirs:
    os.makedirs(dir_path, exist_ok=True)
    print(f"✓ Created: {dir_path}")

# Link output directories
for dir_name in ['preprocessed', 'preprocessed_registered', 'checkpoints', 'visualizations', 'logs']:
    local_path = f'/content/Topo-Brain/{dir_name}'
    drive_path = f'{drive_data}/{dir_name}'
    
    if os.path.exists(local_path) and not os.path.islink(local_path):
        shutil.rmtree(local_path)
    elif os.path.islink(local_path):
        os.remove(local_path)
    
    os.symlink(drive_path, local_path)
    print(f"✓ Linked: {dir_name} -> Google Drive")

print("\n✓ Setup complete! All outputs will be saved to Google Drive.")

## Step 2: Data Preprocessing

**⏱️ Time: ~1-2 hours (depending on dataset size)**

This will:
1. Apply N4 bias correction
2. Perform skull stripping
3. Normalize intensities
4. Save preprocessed volumes

In [ ]:
# Run preprocessing
import sys
sys.path.append('/content/Topo-Brain')

from src.preprocessing import MRIPreprocessor
from src.config import get_default_config
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)

print("Starting preprocessing...")
print("This will take 1-2 hours depending on dataset size.\n")

config = get_default_config()
preprocessor = MRIPreprocessor(config.preprocessing)

# Find all images using discover_dataset
from src.utils import discover_dataset

dataset_info = discover_dataset(
    config.data.data_root,
    config.data
)

print(f"Found {len(dataset_info)} files to preprocess\n")

# Process all files
from tqdm.notebook import tqdm

processed_files = []
for file_info in tqdm(dataset_info, desc="Preprocessing"):
    try:
        input_path = Path(file_info['file_path'])
        
        # Create output path
        relative_path = input_path.relative_to(config.data.data_root)
        output_filename = input_path.name.replace('_defaced', '_preprocessed').replace('.nii.gz', '_preprocessed.nii.gz')
        output_path = config.data.output_root / relative_path.parent / output_filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Preprocess
        result = preprocessor.preprocess_single(input_path, output_path)
        processed_files.append(output_path)
        
    except Exception as e:
        print(f"\n❌ Error processing {file_info['subject']}: {e}")
        continue

print(f"\n✓ Preprocessing complete! Processed {len(processed_files)} files")
print(f"Output saved to: {config.data.output_root}")

## Step 3: 3T→7T Registration (CRITICAL!)

**⏱️ Time: ~30-60 minutes**

**⚠️ DO NOT SKIP!** This is the most important step. Without proper registration, your training WILL fail.

This will align all 3T scans to their corresponding 7T scans using mutual information.

In [ ]:
# Run registration
print("=" * 60)
print("CRITICAL STEP: 3T→7T REGISTRATION")
print("=" * 60)
print("\nThis will align all 3T scans to 7T reference space.")
print("Expected time: 30-60 minutes\n")

!python /content/Topo-Brain/scripts/register_dataset.py \
    --input /content/Topo-Brain/preprocessed \
    --output /content/Topo-Brain/preprocessed_registered \
    --registration-type rigid \
    --log-level INFO

print("\n" + "=" * 60)
print("Registration complete!")
print("=" * 60)

In [ ]:
# Check registration quality
import json
from pathlib import Path

report_path = Path('/content/Topo-Brain/preprocessed_registered/registration_report.json')

if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    
    print("📊 REGISTRATION QUALITY REPORT")
    print("=" * 60)
    
    # Analyze T1w registrations
    t1w_corrs = []
    for subject in report:
        if 'T1w' in subject.get('modalities', {}):
            t1w_data = subject['modalities']['T1w']
            if 'correlation' in t1w_data:
                corr = t1w_data['correlation']
                t1w_corrs.append(corr)
                
                # Color code based on quality
                if corr > 0.7:
                    status = "✅ GOOD"
                elif corr > 0.5:
                    status = "⚠️ REVIEW"
                else:
                    status = "❌ POOR"
                
                print(f"{subject['subject']}: {corr:.3f} {status}")
    
    if t1w_corrs:
        avg_corr = sum(t1w_corrs) / len(t1w_corrs)
        print("\n" + "=" * 60)
        print(f"Average correlation: {avg_corr:.3f}")
        
        good_count = sum(1 for c in t1w_corrs if c > 0.7)
        print(f"Good registrations (>0.7): {good_count}/{len(t1w_corrs)}")
        
        if avg_corr < 0.5:
            print("\n⚠️ WARNING: Low average correlation!")
            print("This may indicate issues with your data.")
            print("Check checkerboard visualizations before training.")
        elif avg_corr > 0.7:
            print("\n✅ Excellent! Registration quality is good.")
            print("You're ready to train!")
    
    print("=" * 60)
else:
    print("❌ Registration report not found!")

In [ ]:
# Visualize registration quality (for one subject)
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

# Pick first subject
subjects = [d for d in Path('/content/Topo-Brain/preprocessed_registered').iterdir() if d.is_dir() and d.name.startswith('sub-')]

if subjects:
    subject = subjects[0]
    print(f"Visualizing registration for: {subject.name}")
    
    # Load registered 3T and 7T
    t1w_3t_path = subject / "ses-1" / "anat" / f"{subject.name}_ses-1_T1w_registered.nii.gz"
    t1w_7t_path = subject / "ses-2" / "anat" / f"{subject.name}_ses-2_T1w_preprocessed.nii.gz"
    
    if t1w_3t_path.exists() and t1w_7t_path.exists():
        img_3t = nib.load(str(t1w_3t_path)).get_fdata()
        img_7t = nib.load(str(t1w_7t_path)).get_fdata()
        
        # Get middle slice
        mid_slice = img_3t.shape[2] // 2
        
        # Create comparison
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(img_3t[:, :, mid_slice].T, cmap='gray', origin='lower')
        axes[0].set_title('3T (Registered)')
        axes[0].axis('off')
        
        axes[1].imshow(img_7t[:, :, mid_slice].T, cmap='gray', origin='lower')
        axes[1].set_title('7T (Target)')
        axes[1].axis('off')
        
        # Difference map
        diff = np.abs(img_3t[:, :, mid_slice] - img_7t[:, :, mid_slice])
        axes[2].imshow(diff.T, cmap='hot', origin='lower')
        axes[2].set_title('Absolute Difference')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig('/content/drive/MyDrive/Topo-Brain-Data/registration_check.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"\n✓ Visualization saved to Google Drive: registration_check.png")
        print("Inspect this to verify alignment quality!")
    else:
        print(f"Files not found for {subject.name}")

## Step 4: Quick Training Test (10 Epochs)

**⏱️ Time: ~2-3 hours**

**Recommended:** Run this first to verify everything works before committing to full training.

This will train for 10 epochs to verify:
- Data loading works correctly
- No CUDA errors
- Discriminator doesn't dominate (D_loss should be 0.3-0.7)
- PSNR shows improvement

In [ ]:
# Quick test run
print("🧪 RUNNING QUICK TEST (10 epochs)")
print("=" * 60)
print("This will verify everything works before full training.")
print("Expected time: 2-3 hours on T4 GPU\n")

# Run quick test
!python /content/Topo-Brain/scripts/train_gan_enhanced.py \
    --epochs 10 \
    --batch-size 2 \
    --lr-g 2e-4 \
    --lr-d 5e-5 \
    --data-root /content/Topo-Brain/preprocessed_registered \
    --checkpoint-dir /content/Topo-Brain/checkpoints_test

print("\n" + "=" * 60)
print("Quick test complete!")
print("=" * 60)
print("\nCheck the output above for:")
print("  ✅ D_loss in range [0.3, 0.7]")
print("  ✅ PSNR increasing")
print("  ✅ No errors or warnings")
print("\nIf all looks good, proceed to full training!")

In [ ]:
# Check test results
import json
from pathlib import Path

checkpoint_dir = Path('/content/Topo-Brain/checkpoints_test')
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob('checkpoint_epoch_*.pth'))
    if checkpoints:
        import torch
        latest = max(checkpoints, key=lambda p: int(p.stem.split('_')[-1]))
        checkpoint = torch.load(latest)
        
        print("📊 QUICK TEST RESULTS")
        print("=" * 60)
        
        if 'val_history' in checkpoint:
            val_psnr = checkpoint['val_history'].get('psnr', [])
            if val_psnr:
                print(f"Best PSNR: {max(val_psnr):.2f} dB")
                print(f"Final PSNR: {val_psnr[-1]:.2f} dB")
        
        if 'train_history' in checkpoint:
            d_loss = checkpoint['train_history'].get('d_loss', [])
            if d_loss:
                final_d_loss = d_loss[-1]
                print(f"Final D_loss: {final_d_loss:.4f}")
                
                if final_d_loss < 0.1:
                    print("\n⚠️ WARNING: D_loss too low!")
                    print("Discriminator is dominating. Check your data registration!")
                elif final_d_loss > 1.5:
                    print("\n⚠️ WARNING: D_loss too high!")
                    print("Generator may be struggling. This is unusual.")
                else:
                    print("\n✅ D_loss in healthy range!")
        
        print("=" * 60)
        
        # Recommend next steps
        if val_psnr and max(val_psnr) > 15:
            print("\n✅ Test successful! Ready for full training.")
            print("Proceed to Step 5 below.")
        else:
            print("\n⚠️ PSNR seems low. Review visualizations and logs.")
            print("Check registration quality before full training.")

## Step 5: Full Training (100 Epochs)

**⏱️ Time: ~35-45 hours on T4 GPU**

**⚠️ Important:** 
- This will run for ~40 hours. Colab may disconnect after 12 hours on free tier.
- Consider Colab Pro for uninterrupted training.
- Alternative: Run in 4-5 sessions, reloading checkpoints each time.

This trains the complete Stage 1 GAN with all enhancements.

In [ ]:
# Full training - Stage 1
print("🚀 STARTING FULL TRAINING - STAGE 1")
print("=" * 60)
print("Training 100 epochs with enhanced GAN")
print("Expected time: 35-45 hours on T4 GPU")
print("=" * 60)
print("\n⚠️ COLAB FREE TIER WARNING:")
print("Colab may disconnect after ~12 hours.")
print("You may need to resume training multiple times.")
print("Progress is saved every 5 epochs.\n")

input("Press Enter to start full training...")

!python /content/Topo-Brain/scripts/train_gan_enhanced.py \
    --epochs 100 \
    --batch-size 2 \
    --lr-g 2e-4 \
    --lr-d 5e-5 \
    --data-root /content/Topo-Brain/preprocessed_registered \
    --checkpoint-dir /content/Topo-Brain/checkpoints_stage1

print("\n✓ Stage 1 training complete!")

In [ ]:
# Resume training if disconnected
import torch
from pathlib import Path

checkpoint_dir = Path('/content/Topo-Brain/checkpoints_stage1')
checkpoints = list(checkpoint_dir.glob('checkpoint_epoch_*.pth'))

if checkpoints:
    latest = max(checkpoints, key=lambda p: int(p.stem.split('_')[-1]))
    checkpoint = torch.load(latest)
    last_epoch = checkpoint['epoch']
    
    print(f"Found checkpoint at epoch {last_epoch}")
    print(f"Resuming training from epoch {last_epoch + 1}")
    
    remaining_epochs = 100 - last_epoch
    
    if remaining_epochs > 0:
        print(f"Remaining epochs: {remaining_epochs}")
        
        !python /content/Topo-Brain/scripts/train_gan_enhanced.py \
            --epochs 100 \
            --batch-size 2 \
            --lr-g 2e-4 \
            --lr-d 5e-5 \
            --data-root /content/Topo-Brain/preprocessed_registered \
            --checkpoint-dir /content/Topo-Brain/checkpoints_stage1 \
            --resume {str(latest)}
    else:
        print("Training already complete!")
else:
    print("No checkpoint found. Start training from Step 5 above.")

## Step 6: Monitor Training Progress

Use these cells to check training status, visualize results, and launch TensorBoard.

In [ ]:
# Load and display training history
import torch
import matplotlib.pyplot as plt
from pathlib import Path

checkpoint_dir = Path('/content/Topo-Brain/checkpoints_stage1')
checkpoints = list(checkpoint_dir.glob('checkpoint_epoch_*.pth'))

if checkpoints:
    latest = max(checkpoints, key=lambda p: int(p.stem.split('_')[-1]))
    checkpoint = torch.load(latest, map_location='cpu')
    
    # Extract history
    train_hist = checkpoint.get('train_history', {})
    val_hist = checkpoint.get('val_history', {})
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Generator loss
    if 'g_loss' in train_hist:
        axes[0, 0].plot(train_hist['g_loss'])
        axes[0, 0].set_title('Generator Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].grid(True)
    
    # Discriminator loss
    if 'd_loss' in train_hist:
        axes[0, 1].plot(train_hist['d_loss'])
        axes[0, 1].axhline(y=0.3, color='g', linestyle='--', label='Target min')
        axes[0, 1].axhline(y=0.7, color='g', linestyle='--', label='Target max')
        axes[0, 1].set_title('Discriminator Loss (should be 0.3-0.7)')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # Validation PSNR
    if 'psnr' in val_hist:
        axes[1, 0].plot(val_hist['psnr'])
        axes[1, 0].set_title('Validation PSNR (higher is better)')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('dB')
        axes[1, 0].grid(True)
    
    # L1 Loss
    if 'g_l1_loss' in train_hist:
        axes[1, 1].plot(train_hist['g_l1_loss'])
        axes[1, 1].set_title('L1 Reconstruction Loss')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Topo-Brain-Data/training_curves.png', dpi=150)
    plt.show()
    
    # Summary
    print("\n📊 TRAINING SUMMARY")
    print("=" * 60)
    print(f"Epochs completed: {checkpoint['epoch']}")
    if 'psnr' in val_hist and val_hist['psnr']:
        print(f"Best PSNR: {max(val_hist['psnr']):.2f} dB")
        print(f"Current PSNR: {val_hist['psnr'][-1]:.2f} dB")
    if 'd_loss' in train_hist and train_hist['d_loss']:
        print(f"Current D_loss: {train_hist['d_loss'][-1]:.4f}")
    print("=" * 60)
else:
    print("No checkpoints found. Training hasn't started yet.")

In [ ]:
# Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/Topo-Brain/logs_enhanced

In [ ]:
# View latest visualizations
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt

vis_dir = Path('/content/Topo-Brain/visualizations_enhanced')

if vis_dir.exists():
    # Find latest epoch
    epoch_dirs = sorted([d for d in vis_dir.iterdir() if d.is_dir()])
    
    if epoch_dirs:
        latest_epoch = epoch_dirs[-1]
        print(f"📸 Visualizations from: {latest_epoch.name}")
        print("=" * 60)
        
        # Show first few samples
        images = sorted(list(latest_epoch.glob('*.png')))[:6]
        
        if images:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.flatten()
            
            for idx, img_path in enumerate(images[:6]):
                img = plt.imread(str(img_path))
                axes[idx].imshow(img)
                axes[idx].set_title(img_path.name)
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print("No images found")
    else:
        print("No epoch directories found")
else:
    print("Visualization directory doesn't exist yet")

## Step 7: Evaluate Results

After training completes, evaluate the model performance.

In [ ]:
# Evaluate on test set
print("📊 FINAL EVALUATION")
print("=" * 60)

!python /content/Topo-Brain/scripts/eval_gan.py \
    --checkpoint /content/Topo-Brain/checkpoints_stage1/best_model.pth \
    --data-root /content/Topo-Brain/preprocessed_registered \
    --output-dir /content/Topo-Brain/evaluation_results

print("\n✓ Evaluation complete!")
print("Results saved to: /content/Topo-Brain/evaluation_results")

## Step 8: Download Trained Models

Download your trained models to use locally or for inference.

In [ ]:
# Check model files in Google Drive
from pathlib import Path

checkpoint_dir = Path('/content/drive/MyDrive/Topo-Brain-Data/checkpoints')

print("📦 AVAILABLE CHECKPOINTS")
print("=" * 60)

for checkpoint_folder in ['checkpoints_test', 'checkpoints_stage1', 'checkpoints_full']:
    folder = checkpoint_dir.parent / checkpoint_folder
    if folder.exists():
        print(f"\n{checkpoint_folder}/")
        files = list(folder.glob('*.pth'))
        for f in files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  {f.name} ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("All checkpoints are saved in your Google Drive!")
print("You can download them directly from Google Drive.")

## 🎓 Summary & Next Steps

**What you've accomplished:**
✅ Preprocessed MRI data with N4 correction and skull stripping  
✅ Registered all 3T-7T pairs with quality validation  
✅ Trained enhanced GAN with topology-aware losses  
✅ Monitored training with TensorBoard  
✅ Evaluated model performance  

**Next steps:**

1. **Stage 2 (Optional):** Train Persistent Homology Refiner
2. **Stage 3 (Optional):** Train Latent Diffusion Model  
3. **Inference:** Run model on new 3T scans
4. **Analysis:** Compare with baseline methods

**For full pipeline:**
```python
!python /content/Topo-Brain/scripts/run_full_pipeline.py \
    --mode train \
    --data-root /content/Topo-Brain/preprocessed_registered \
    --epochs-stage1 100 \
    --epochs-stage2 50 \
    --epochs-stage3 100
```

---

## 🆘 Troubleshooting

**Session disconnected during training?**
- Checkpoints are saved every 5 epochs to Google Drive
- Re-run the "Resume training" cell in Step 5

**Out of memory?**
- Reduce batch size: `--batch-size 1`
- Reduce patch size in config

**Low PSNR (<15 dB)?**
- Check registration quality report
- Verify data alignment visually
- Re-run registration if needed

**For more help:** See [QUICKSTART.md](QUICKSTART.md) in the repository

---

**Good luck with your training!** 🚀